In [1]:
from agentics import Agentics as AG
from crewai_tools import MCPServerAdapter
from mcp import StdioServerParameters # For Stdio Server
import os
from agentics.core.llm_connections import available_llms
from pydantic import BaseModel, Field
from typing import Optional
from dotenv import load_dotenv
import asyncio

2025-09-02 03:39:20.552 | DEBUG    | agentics.core.llm_connections:<module>:66 - AGENTICS is connecting to the following LLM API providers:
2025-09-02 03:39:20.553 | DEBUG    | agentics.core.llm_connections:<module>:69 - 0 - WatsonX
2025-09-02 03:39:20.553 | DEBUG    | agentics.core.llm_connections:<module>:74 - 1 - Gemini
2025-09-02 03:39:20.553 | DEBUG    | agentics.core.llm_connections:<module>:78 - 2 - OpenAI
2025-09-02 03:39:20.553 | DEBUG    | agentics.core.llm_connections:<module>:80 - Please add API keys in .env file to add or disconnect providers.
/Users/boxuanli/Documents/Code/agentics/.venv/lib/python3.12/site-packages/pydantic/_internal/_config.py:323: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  warnings.warn(DEPRECATION_MESSAGE, DeprecationWarning)


2025-09-02 03:40:25,052 - 6212808704 - telemetry.py-telemetry:51 - ERROR: HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Max retries exceeded with url: /v1/traces (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x1680cf260>, 'Connection to telemetry.crewai.com timed out. (connect timeout=29.999993085861206)'))
2025-09-02 03:41:30,303 - 6212808704 - telemetry.py-telemetry:51 - ERROR: HTTPSConnectionPool(host='telemetry.crewai.com', port=4319): Max retries exceeded with url: /v1/traces (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x1686988c0>, 'Connection to telemetry.crewai.com timed out. (connect timeout=29.99998903274536)'))


In [2]:

class Stock(BaseModel):
    ticker:str
    news:Optional [str]=None
    sentiment: Optional [str]=None
class Question(BaseModel):
    question :Optional[str] = None
class Rec(BaseModel):
    buyorsell: Optional[str] = None
    reason:Optional[str] =None


In [ ]:
from pydantic import BaseModel, Field
from agentics.core.llm_connections import  gemini_llm

portfolio = AG(atype=Stock)
tickers = ["AAPL","NVDA"]

for t in tickers:
    portfolio.states.append(Stock(ticker=t))

for s in portfolio:  
    print(s.model_dump_json())

{"ticker":"AAPL","news":null,"sentiment":null}
{"ticker":"NVDA","news":null,"sentiment":null}


In [9]:
from ddgs import DDGS
async def get_news(state: Stock) -> Stock:
    state.news=str(DDGS().text(f"News related to the market sentiment of stock {state.ticker} today", max_results=1))
    return state    
portfolio = await portfolio.amap(get_news)

2025-09-02 03:39:50.546 | DEBUG    | agentics.core.agentics:amap:206 - Executing amap on function <function get_news at 0x168223240>
2025-09-02 03:39:52.386 | DEBUG    | agentics.core.agentics:amap:231 - 2 states processed. 0.18389008045196534 seconds average per state in the last chunk ...


In [10]:
print(portfolio.pretty_print())

ticker: AAPL
news: "[{'title': '\xBB 2022 \xBB April PRACTICAL STOCK INVESTING', 'href': 'https://practicalstockinvesting.com/2022/04/',\
  \ 'body': '... to what degree can we depend on reading stock prices ... In any event,\
  \ AAPL gave the overnight market an excuse (not a reason) to sell off it wanted.'}]"
sentiment: null

ticker: NVDA
news: '[{''title'': ''Nvidia stock in focus amid post-earnings dip ( NVDA ...) | Seeking
  Alpha'', ''href'': ''https://seekingalpha.com/news/4490633-nvidia-stock-focus-amid-post-earnings-dip'',
  ''body'': ''Nvidia ( NVDA ) stock draws a wide range of views from Seeking Alpha
  analysts after shares of the chipmaker fell following its Q2 fiscal 2026 results
  this week. Read more here.''}]'
sentiment: null


ticker: AAPL
news: "[{'title': '\xBB 2022 \xBB April PRACTICAL STOCK INVESTING', 'href': 'https://practicalstockinvesting.com/2022/04/',\
  \ 'body': '... to what degree can we depend on reading stock prices ... In any event,\
  \ AAPL gave th

In [11]:
portfolio.llm = gemini_llm

#extended_porfolio = portfolio.add_attribute("recommendation", slot_type="str", description="Sentiment towards the stock")
analyzed_portfolio = await portfolio.self_transduction(
        source_fields=["ticker", "news"], 
        target_fields=["sentiment"],
        instructions="Generatea the sentiment towards the stock"
    )
print(analyzed_portfolio.pretty_print())
analyzed_portfolio.to_csv("sentiment.csv")

2025-09-02 03:39:52.396 | DEBUG    | agentics.core.agentics:__lshift__:518 - Executing task: Generatea the sentiment towards the stock
2 states will be transduced
2025-09-02 03:39:52.396 | DEBUG    | agentics.core.agentics:__lshift__:612 - transducer class: <class 'agentics.abstractions.pydantic_transducer.PydanticTransducerCrewAI'>
2025-09-02 03:39:53.687 | DEBUG    | agentics.core.agentics:__lshift__:648 - Processed 2 states in 1.2899556159973145 seconds
2025-09-02 03:39:53.687 | DEBUG    | agentics.core.agentics:__lshift__:700 - 2 states processed in 0.06449778079986572 seconds average per state ...
2025-09-02 03:39:53.689 | DEBUG    | agentics.core.agentics:to_csv:928 - Exporting 2 Agentics to CSV sentiment.csv


ticker: AAPL
news: "[{'title': '\xBB 2022 \xBB April PRACTICAL STOCK INVESTING', 'href': 'https://practicalstockinvesting.com/2022/04/',\
  \ 'body': '... to what degree can we depend on reading stock prices ... In any event,\
  \ AAPL gave the overnight market an excuse (not a reason) to sell off it wanted.'}]"
sentiment: negative

ticker: NVDA
news: '[{''title'': ''Nvidia stock in focus amid post-earnings dip ( NVDA ...) | Seeking
  Alpha'', ''href'': ''https://seekingalpha.com/news/4490633-nvidia-stock-focus-amid-post-earnings-dip'',
  ''body'': ''Nvidia ( NVDA ) stock draws a wide range of views from Seeking Alpha
  analysts after shares of the chipmaker fell following its Q2 fiscal 2026 results
  this week. Read more here.''}]'
sentiment: Neutral


ticker: AAPL
news: "[{'title': '\xBB 2022 \xBB April PRACTICAL STOCK INVESTING', 'href': 'https://practicalstockinvesting.com/2022/04/',\
  \ 'body': '... to what degree can we depend on reading stock prices ... In any event,\
  \ AAPL 

In [12]:

from agentics import Agentics as AG
from pydantic import BaseModel
from typing import Optional

for s in portfolio:
    source = AG(atype=Question,states=[Question(question="Tell me if I should buy or sell" +s.ticker+" and justfity based on"+s.news) ])
    target= AG(atype=Rec,
        llm=gemini_llm,
        verbose_agent=False) 
    answer = await (target << source)
    print(answer.pretty_print())


2025-09-02 03:39:53.696 | DEBUG    | agentics.core.agentics:__lshift__:518 - Executing task: Generate an object of the specified type from the following input.
1 states will be transduced
2025-09-02 03:39:53.697 | DEBUG    | agentics.core.agentics:__lshift__:612 - transducer class: <class 'agentics.abstractions.pydantic_transducer.PydanticTransducerCrewAI'>
2025-09-02 03:39:55.034 | DEBUG    | agentics.core.agentics:__lshift__:648 - Processed 1 states in 1.3353097438812256 seconds
2025-09-02 03:39:55.034 | DEBUG    | agentics.core.agentics:__lshift__:700 - 1 states processed in 0.06676548719406128 seconds average per state ...
2025-09-02 03:39:55.035 | DEBUG    | agentics.core.agentics:__lshift__:518 - Executing task: Generate an object of the specified type from the following input.
1 states will be transduced
2025-09-02 03:39:55.035 | DEBUG    | agentics.core.agentics:__lshift__:612 - transducer class: <class 'agentics.abstractions.pydantic_transducer.PydanticTransducerCrewAI'>


buyorsell: null
reason: AAPL gave the overnight market an excuse (not a reason) to sell off it wanted.


buyorsell: null
reason: AAPL gave the overnight market an excuse (not a reason) to sell off it wanted.




2025-09-02 03:39:56.586 | DEBUG    | agentics.core.agentics:__lshift__:648 - Processed 1 states in 1.5501492023468018 seconds
2025-09-02 03:39:56.586 | DEBUG    | agentics.core.agentics:__lshift__:700 - 1 states processed in 0.07750746011734008 seconds average per state ...


buyorsell: null
reason: Analyst opinions are mixed following a post-earnings dip in Nvidia's stock
  price.


buyorsell: null
reason: Analyst opinions are mixed following a post-earnings dip in Nvidia's stock
  price.


